# Summary_Day11_offline.ipynb  
## CNN 기반 이미지 분류 · 인터넷 불가 버전 · 합성 이미지 데이터로 CNN 구조 연습

이 파일은 **인터넷이 안 되는 환경**에서 11강 CNN 흐름을 연습하기 위한 버전이다.

원본 강의는 MNIST와 CIFAR-10을 다운로드해서 사용한다.  
하지만 인터넷이 없으면 `datasets.MNIST(download=True)`나 `datasets.CIFAR10(download=True)`가 실패할 수 있다.

그래서 이 파일은 다운로드 없이 실행되도록 직접 작은 합성 이미지 데이터를 만든다.

```text
MNIST 대신: 28×28 선 이미지
CIFAR-10 대신: 3×32×32 컬러 패턴 이미지
```

데이터는 다르지만 핵심 학습 흐름은 강의와 같다.

```text
이미지 4D Tensor
→ Conv2d
→ ReLU
→ MaxPool2d
→ Flatten
→ Linear
→ CrossEntropyLoss
→ FCN과 CNN 비교
```

> 필기 포인트:  
> 인터넷이 없을 때는 데이터셋 다운로드가 막힐 수 있다.  
> 그래도 CNN의 shape 흐름과 코드 패턴은 합성 이미지로 충분히 연습할 수 있다.

## 1. 전체 실습 목적

이 노트북의 목적은 다음이다.

1. 다운로드 없이 CNN 구조를 실행한다.
2. `[N, C, H, W]` 이미지 Tensor를 직접 만든다.
3. `Conv2d`가 feature map을 만드는 흐름을 확인한다.
4. FCN은 이미지를 펼쳐서 학습하고, CNN은 공간 구조를 유지한다는 차이를 본다.
5. 같은 합성 이미지 데이터를 FCN과 CNN으로 학습해 비교한다.
6. `view`, `permute`, `Flatten`의 역할을 정리한다.

## 2. 라이브러리 준비

인터넷 불가 버전은 `torchvision.datasets` 다운로드를 사용하지 않는다.  
순수 NumPy와 PyTorch로 합성 데이터를 만든다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix

%matplotlib inline

torch.manual_seed(123)
np.random.seed(123)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("device:", device)
print("PyTorch:", torch.__version__)

## 3. ReLU 복습

CNN에서도 ReLU는 Conv2d 뒤에 자주 들어간다.

In [ ]:
relu = nn.ReLU()

x = torch.linspace(-2, 2, 50)
y = relu(x)

plt.plot(x.numpy(), y.numpy())
plt.xlabel("x")
plt.ylabel("ReLU(x)")
plt.title("ReLU Function")
plt.show()

## 4. 합성 MNIST식 이미지 만들기

Conv2d 감각을 보기 위해 28×28 흑백 이미지를 직접 만든다.

대각선과 세로선이 있는 간단한 이미지다.

```text
shape: [1, 28, 28]
Conv2d 입력용 shape: [1, 1, 28, 28]
```

In [ ]:
image_np = np.zeros((28, 28), dtype=np.float32)

for i in range(5, 23):
    image_np[i, i] = 1.0
    image_np[i, 14] = 0.8

image_tensor = torch.tensor(image_np).view(1, 1, 28, 28)

print("image_tensor shape:", image_tensor.shape)

plt.imshow(image_np, cmap="gray_r")
plt.title("Synthetic 28x28 Image")
plt.axis("off")
plt.show()

## 5. 대각선 필터 Conv2d 적용

강의와 같은 대각선 필터를 만든다.

```python
nn.Conv2d(1, 1, 3)
```

- 입력 channel 1개다.
- 출력 feature map 1개다.
- kernel 크기는 3×3이다.

In [ ]:
conv_diag = nn.Conv2d(1, 1, 3)

nn.init.constant_(conv_diag.bias, 0.0)

w_np = np.array([
    [0, 0, 1],
    [0, 1, 0],
    [1, 0, 0]
], dtype=np.float32)

conv_diag.weight.data = torch.tensor(w_np).view(1, 1, 3, 3)

w1 = conv_diag(image_tensor)
w2 = conv_diag(w1)
w3 = conv_diag(w2)

conv_images = [image_tensor, w1, w2, w3]

for i, img in enumerate(conv_images):
    print(i, img.shape)

In [ ]:
plt.figure(figsize=(8, 2))

for i, img in enumerate(conv_images):
    ax = plt.subplot(1, 4, i + 1)
    arr = img.detach().numpy().reshape(img.shape[-2], img.shape[-1])
    plt.imshow(arr, cmap="gray_r")
    plt.title(f"step {i}")
    plt.axis("off")

plt.tight_layout()
plt.show()

그래프 해석:

- 필터가 반응하는 방향의 선이 강조된다.
- padding이 없기 때문에 통과할 때마다 이미지 크기가 줄어든다.
- Conv2d 결과가 feature map이다.

## 6. Conv2d / ReLU / MaxPool2d shape 확인

CIFAR-10과 같은 RGB 이미지 shape을 가정한다.

```text
입력: [100, 3, 32, 32]
출력: [100, 32, 14, 14]
```

In [ ]:
conv1 = nn.Conv2d(3, 32, 3)
relu = nn.ReLU(inplace=True)
conv2 = nn.Conv2d(32, 32, 3)
maxpool = nn.MaxPool2d((2, 2))

dummy = torch.randn(100, 3, 32, 32)

x1 = conv1(dummy)
x2 = relu(x1)
x3 = conv2(x2)
x4 = relu(x3)
x5 = maxpool(x4)

print("dummy:", dummy.shape)
print("conv1:", x1.shape)
print("relu1:", x2.shape)
print("conv2:", x3.shape)
print("relu2:", x4.shape)
print("maxpool:", x5.shape)

## 7. Flatten 확인

`[100, 32, 14, 14]`를 Linear에 넣으려면 `[100, 6272]`로 펼쳐야 한다.

In [ ]:
flatten = nn.Flatten()
flat = flatten(x5)

print("Flatten 이전:", x5.shape)
print("Flatten 이후:", flat.shape)
print("32*14*14:", 32 * 14 * 14)

## 8. 합성 CIFAR-10식 데이터 만들기

인터넷 없이 CIFAR-10 구조를 흉내 낸 데이터를 만든다.

```text
이미지 shape: [3, 32, 32]
class 수: 10개
```

각 class마다 색상과 선 위치가 다르게 들어가도록 만든다.  
이렇게 하면 CNN과 FCN이 분류 구조를 학습할 수 있다.

In [ ]:
def make_synthetic_cifar_like(n_per_class=120, noise_level=0.12):
    images = []
    labels = []

    for cls in range(10):
        for _ in range(n_per_class):
            img = np.random.normal(0.0, noise_level, size=(3, 32, 32)).astype(np.float32)

            channel = cls % 3
            row = 4 + (cls * 2) % 20
            col = 4 + (cls * 3) % 20

            img[channel, row:row + 6, :] += 1.0
            img[channel, :, col:col + 4] += 0.7

            if cls % 2 == 0:
                for i in range(8, 24):
                    img[channel, i, i] += 0.8
            else:
                for i in range(8, 24):
                    img[channel, i, 31 - i] += 0.8

            img = np.clip(img, -1.0, 1.0)

            images.append(img)
            labels.append(cls)

    X = np.stack(images)
    y = np.array(labels)

    indices = np.random.permutation(len(y))

    return X[indices], y[indices]

X, y = make_synthetic_cifar_like(n_per_class=120)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("class 분포:", np.bincount(y))

## 9. 합성 이미지 시각화

PyTorch 이미지는 CHW 순서라서 `imshow`에 넣으려면 HWC로 바꿔야 한다.

### 함수 사용법

```python
img.transpose(1, 2, 0)
```

- NumPy 배열에서 CHW를 HWC로 바꾼다.
- PyTorch Tensor에서는 `permute(1, 2, 0)`을 쓴다.

In [ ]:
plt.figure(figsize=(10, 4))

for i in range(20):
    ax = plt.subplot(2, 10, i + 1)
    img = X[i].transpose(1, 2, 0)
    img = (img + 1.0) / 2.0
    img = np.clip(img, 0, 1)

    plt.imshow(img)
    plt.title(str(y[i]), fontsize=8)
    plt.axis("off")

plt.tight_layout()
plt.show()

## 10. Train / Test 분할

직접 index를 나누어 train/test를 만든다.

In [ ]:
n_total = len(y)
n_train = int(n_total * 0.8)

X_train = X[:n_train]
y_train = y[:n_train]

X_test = X[n_train:]
y_test = y[n_train:]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

## 11. FCN용 데이터와 CNN용 데이터 준비

FCN은 이미지를 펼쳐서 사용한다.

```text
CNN용: [N, 3, 32, 32]
FCN용: [N, 3072]
```

In [ ]:
X_train_cnn = torch.tensor(X_train).float()
X_test_cnn = torch.tensor(X_test).float()

X_train_fcn = X_train_cnn.view(len(X_train_cnn), -1)
X_test_fcn = X_test_cnn.view(len(X_test_cnn), -1)

y_train_t = torch.tensor(y_train).long()
y_test_t = torch.tensor(y_test).long()

print("X_train_cnn:", X_train_cnn.shape)
print("X_train_fcn:", X_train_fcn.shape)
print("y_train_t:", y_train_t.shape)

## 12. TensorDataset과 DataLoader

다운로드형 Dataset이 없을 때는 `TensorDataset`을 사용한다.

In [ ]:
batch_size = 64

train_loader_fcn = DataLoader(
    TensorDataset(X_train_fcn, y_train_t),
    batch_size=batch_size,
    shuffle=True
)

test_loader_fcn = DataLoader(
    TensorDataset(X_test_fcn, y_test_t),
    batch_size=batch_size,
    shuffle=False
)

train_loader_cnn = DataLoader(
    TensorDataset(X_train_cnn, y_train_t),
    batch_size=batch_size,
    shuffle=True
)

test_loader_cnn = DataLoader(
    TensorDataset(X_test_cnn, y_test_t),
    batch_size=batch_size,
    shuffle=False
)

print("FCN batch:", len(train_loader_fcn))
print("CNN batch:", len(train_loader_cnn))

## 13. 공통 학습 함수 정의

FCN과 CNN을 같은 함수로 학습한다.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        pred = torch.max(outputs, 1)[1]

        total_loss += loss.item() * y_batch.size(0)
        correct += (pred == y_batch).sum().item()
        total += y_batch.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    y_true = []
    y_pred = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            pred = torch.max(outputs, 1)[1]

            total_loss += loss.item() * y_batch.size(0)
            correct += (pred == y_batch).sum().item()
            total += y_batch.size(0)

            y_true.extend(y_batch.cpu().numpy())
            y_pred.extend(pred.cpu().numpy())

    return total_loss / total, correct / total, np.array(y_true), np.array(y_pred)


def train_model(model, train_loader, test_loader, criterion, optimizer, device, num_epochs=5):
    history = []

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        test_loss, test_acc, _, _ = evaluate(model, test_loader, criterion, device)

        history.append([epoch + 1, train_loss, train_acc, test_loss, test_acc])

        print(
            f"epoch {epoch + 1} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"test_loss={test_loss:.4f} | test_acc={test_acc:.4f}"
        )

    return np.array(history)

## 14. FCN 모델 정의

FCN은 이미지를 펼친 `[3072]` 벡터를 입력으로 받는다.

In [ ]:
class FCNNet(nn.Module):
    def __init__(self, n_input=3072, n_hidden=128, n_output=10):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(n_input, n_hidden),
            nn.ReLU(inplace=True),
            nn.Linear(n_hidden, n_output)
        )

    def forward(self, x):
        return self.network(x)

fcn_model = FCNNet().to(device)

criterion = nn.CrossEntropyLoss()
optimizer_fcn = optim.SGD(fcn_model.parameters(), lr=0.05)

print(fcn_model)

## 15. FCN 모델 학습

이미지를 1차원으로 펼친 상태로 학습한다.

In [ ]:
history_fcn = train_model(
    fcn_model,
    train_loader_fcn,
    test_loader_fcn,
    criterion,
    optimizer_fcn,
    device,
    num_epochs=5
)

## 16. CNN 모델 정의

CNN은 이미지를 `[3, 32, 32]` 형태 그대로 입력받는다.

In [ ]:
class CNNNet(nn.Module):
    def __init__(self, n_hidden=128, n_output=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3),
            nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 2))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 14 * 14, n_hidden),
            nn.ReLU(inplace=True),
            nn.Linear(n_hidden, n_output)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

cnn_model = CNNNet().to(device)

criterion = nn.CrossEntropyLoss()
optimizer_cnn = optim.SGD(cnn_model.parameters(), lr=0.05)

print(cnn_model)

## 17. CNN 모델 shape 확인

dummy input으로 CNN 내부 shape을 확인한다.

In [ ]:
dummy = torch.randn(64, 3, 32, 32).to(device)

with torch.no_grad():
    features = cnn_model.features(dummy)
    outputs = cnn_model(dummy)

print("dummy:", dummy.shape)
print("features:", features.shape)
print("outputs:", outputs.shape)

## 18. CNN 모델 학습

CNN은 공간 구조를 유지하면서 학습한다.

In [ ]:
history_cnn = train_model(
    cnn_model,
    train_loader_cnn,
    test_loader_cnn,
    criterion,
    optimizer_cnn,
    device,
    num_epochs=5
)

## 19. FCN과 CNN 비교 그래프

합성 데이터에서는 패턴이 단순해서 두 모델 모두 잘 맞을 수 있다.  
그래도 입력 구조 차이는 명확하다.

```text
FCN: [N, 3072]
CNN: [N, 3, 32, 32]
```

In [ ]:
plt.plot(history_fcn[:, 0], history_fcn[:, 4], label="FCN test acc")
plt.plot(history_cnn[:, 0], history_cnn[:, 4], label="CNN test acc")
plt.xlabel("epoch")
plt.ylabel("test accuracy")
plt.title("FCN vs CNN on Synthetic Images")
plt.legend()
plt.show()

print("FCN final acc:", history_fcn[-1, 4])
print("CNN final acc:", history_cnn[-1, 4])

## 20. CNN 결과 평가

classification report와 confusion matrix를 확인한다.

In [ ]:
test_loss, test_acc, y_true, y_pred = evaluate(
    cnn_model,
    test_loader_cnn,
    criterion,
    device
)

print("test accuracy:", test_acc)

target_names = [f"class {i}" for i in range(10)]

print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
print(cm)

In [ ]:
plt.imshow(cm)
plt.title("Synthetic CNN Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

for (i, j), value in np.ndenumerate(cm):
    if value > 0:
        plt.text(j, i, str(value), ha="center", va="center", fontsize=8)

plt.colorbar()
plt.show()

## 21. 예측 이미지 확인

예측이 맞으면 검정색, 틀리면 빨간색으로 표시한다.

In [ ]:
def show_predictions(model, X_tensor, y_tensor, device, n_show=20):
    model.eval()

    with torch.no_grad():
        inputs = X_tensor[:n_show].to(device)
        outputs = model(inputs)
        preds = torch.max(outputs, 1)[1].cpu()

    plt.figure(figsize=(10, 4))

    for i in range(n_show):
        ax = plt.subplot(2, 10, i + 1)

        img = X_tensor[i].permute(1, 2, 0).numpy()
        img = (img + 1.0) / 2.0
        img = np.clip(img, 0, 1)

        true_label = y_tensor[i].item()
        pred_label = preds[i].item()

        plt.imshow(img)
        plt.title(f"{true_label}→{pred_label}", color="black" if true_label == pred_label else "red", fontsize=8)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_predictions(cnn_model, X_test_cnn, y_test_t, device, n_show=20)

## 22. view와 permute 연습

강의 마지막의 Tensor 변환 연습을 다시 해 본다.

In [ ]:
sample = torch.tensor(np.array([
    [[0, 0, 1],
     [0, 1, 0],
     [1, 0, 0]]
])).float()

print("sample shape:", sample.shape)
print("view(-1) shape:", sample.view(-1).shape)
print("permute(1,2,0) shape:", sample.permute(1, 2, 0).shape)
print("ndim:", sample.ndim)

## 23. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `Conv2d` | 2차원 합성곱 | `nn.Conv2d(in_ch, out_ch, kernel)` |
| `kernel` | 필터 행렬 | 특징을 뽑는 작은 창 |
| `feature map` | Conv 결과 | 필터가 반응한 결과 |
| `MaxPool2d` | 최대 풀링 | 크기 축소 |
| `Flatten` | 1차원 펼치기 | CNN 특징을 Linear에 넣기 |
| `FCN` | 완전 결합형 모델 | 이미지를 펼쳐서 학습 |
| `CNN` | 합성곱 신경망 | 이미지 구조 유지 |
| `TensorDataset` | Tensor 데이터셋 | 인터넷 없이 데이터 구성 |
| `DataLoader` | mini-batch 도구 | batch 단위 학습 |
| `view` | shape 변경 | Flatten 등에 사용 |
| `permute` | 차원 순서 변경 | CHW ↔ HWC |
| `NCHW` | CNN 입력 순서 | batch, channel, height, width |
| `CrossEntropyLoss` | 다중 분류 손실 | logits + long label |

## 24. 시험용 요약

```text
인터넷 불가 버전의 핵심 = 합성 이미지로 CNN shape과 학습 구조를 연습한다
```

꼭 기억할 것:

- 원본 11강은 MNIST와 CIFAR-10을 사용한다.
- 인터넷이 없으면 다운로드 데이터셋이 실패할 수 있다.
- 합성 이미지로도 CNN의 shape 흐름은 충분히 연습할 수 있다.
- CNN 입력은 `[N, C, H, W]`다.
- RGB 이미지는 channel이 3개다.
- FCN은 이미지를 `[N, 3072]`로 펼쳐서 학습한다.
- CNN은 이미지를 `[N, 3, 32, 32]`로 유지해서 학습한다.
- Conv2d는 필터를 움직여 feature map을 만든다.
- MaxPool2d는 공간 크기를 줄인다.
- Flatten은 CNN 특징을 Linear에 넣기 위해 1차원으로 펼친다.
- `32 × 14 × 14 = 6272`는 이번 CNN classifier의 입력 크기다.
- `permute(1,2,0)`은 CHW를 HWC로 바꿀 때 사용한다.
- `CrossEntropyLoss`에는 Softmax를 먼저 적용하지 않는다.
- 예측 class는 `torch.max(outputs, 1)[1]`로 구한다.